# Baseline vs a case: production, value and who pays whom

Kernel: **alvheim-tiein (study .venv)** (`studies/alvheim-tiein/.venv`; `uv sync` there first). The repo's own environment lacks pandas and the study package.

Pick a **project** and a case to compare with its baseline. With `DEMO = True` this runs on
**synthetic results and fictional economics** (the `demo` project: `results-workbench`'s
example study, whose infill wells play the new reservoir; companies, shares, prices and
tariffs from `examples/demo`).

In [ ]:
DEMO = True
PROJECT = None   # project slug; None = the current one
CASE = None      # case to compare with the baseline; None = the first switched-on (else other) case
UNCAPPED = None  # optional: the case with facility limits relaxed (e.g. "tie_in_uncapped")

In [ ]:
import pandas as pd
import plotly.express as px

from alvheim_tiein import report
from alvheim_tiein import settlement as s
from alvheim_tiein.config import Settings
from alvheim_tiein.demo import DEMO_WITH, build_demo_data
from alvheim_tiein.economics import Reference, value_volumes
from alvheim_tiein.project import Workspace
from alvheim_tiein.results import (
    export_workbench_folder,
    load_task_results,
    period_volumes,
    volume_convention_check,
)

settings = Settings.from_env()
ws = Workspace(build_demo_data(settings.data_dir / "demo") if DEMO else settings.data_dir)
data = report.load_project(ws, "demo" if DEMO else PROJECT)
cases = data.runs
done = data.completed()
BASE = data.baseline_name
switched_on = done.loc[done["switch"] == "on", "case_name"].tolist()
others = [c for c in done["case_name"] if c != BASE]
ALT = DEMO_WITH if DEMO else (CASE or (switched_on or others or [None])[0])
USERS = data.project.analysis.user_licenses
banner = "DEMO: synthetic results, fictional economics\n" if DEMO else ""
print(f"{banner}{data.project.name}: {BASE!r} -> {ALT!r}; paying licence(s) {USERS}")
cases

In [ ]:
results = load_task_results(done.to_dict("records"), data.paths.results)
volume_convention_check(results)

The check compares volumes from rates with cumulative-oil increments. Volumes below use the
`forward` convention (a timestep's rate holds until the next timestep). If `backward` fits a
real model clearly better, change `period_volumes` before trusting the numbers.

In [ ]:
ref = Reference.load(ws.reference_dir(data.paths))
valued = value_volumes(period_volumes(results), ref)
CUR = ref.economics.currency
valued.groupby("case_name")[["oil_sm3", "gas_sm3", "water_sm3", "pv"]].sum().round(0)

## 1. What changes, well by well

In [ ]:
wells = s.compare(valued, BASE, ALT, by=("field", "license", "identifier"))
wells["delta_oil_ksm3"] = wells["delta_oil_sm3"] / 1e3
px.bar(
    wells.sort_values("delta_oil_sm3"), x="identifier", y="delta_oil_ksm3", color="license",
    title=f"Oil, {ALT} minus {BASE} (kSm3)",
).show()
wells[["field", "license", "identifier", "delta_oil_sm3", "delta_gas_sm3", "delta_water_sm3", "delta_pv"]].round(0)

In [ ]:
monthly = s.totals(valued[valued["case_name"].isin([BASE, ALT])], by=("license", "period"), values=("oil_sm3",))
wide = monthly.pivot_table(index=["license", "period"], columns="case_name", values="oil_sm3").reset_index()
wide["delta_oil_sm3"] = wide[ALT] - wide[BASE]
px.line(
    wide, x="period", y="delta_oil_sm3", color="license",
    title="Monthly oil change per licence (Sm3). Deferral = a dip later recovered; loss = never recovered",
).show()

In [ ]:
lic = s.compare(valued, BASE, ALT, by=("license",))
lic[["license", "delta_oe_sm3", "delta_cash_flow", "delta_pv"]].round(0)

`delta_cash_flow` is undiscounted, `delta_pv` discounted. A licence whose undiscounted change is
near zero but whose PV falls had production **deferred**; if both fall, production was **lost**
within the simulated horizon. Volumes deferred past the end of the simulation look lost, so
the horizon must run to the end of field life for this distinction to hold.

In [ ]:
if UNCAPPED and UNCAPPED in set(done["case_name"]):
    display(s.decompose(valued, BASE, ALT, UNCAPPED, by=("license",)).round(0))
else:
    print("No uncapped case: the back-pressure vs capacity-displacement split needs one (set UNCAPPED)")

## 2. Who pays whom

First-model rules (see `settlement.py`): the paying licence(s) compensate every other licence
for its lost value (PV), and pay the change in tariffs. Licence flows are spread over both
sides' company shares. The diagonal of the matrix is a company paying itself through
different licences.

In [ ]:
st = s.settle(valued, ref, BASE, ALT, USERS)
st.licence_flows.round(0)

In [ ]:
print(f"payer (rows) -> payee (columns), million {CUR}")
(st.company_matrix / 1e6).round(2)

In [ ]:
net = st.net.rename_axis("company").reset_index()
px.bar(net, x="company", y="net_received", title=f"Net received from settlement ({CUR}; negative = pays)").show()
net.round(0)

## 3. Hub value vs company value

Settlement payments cancel out in total hub value, so the hub's best case does not depend on
them; each company's best case can. The price of conflict is what the hub loses if one
company's preferred case were chosen.

In [ ]:
table = s.objective_table(valued, ref)
value_cols = ["hub", *[c for c in ref.companies() if c in table]]
display((table[value_cols] / 1e6).round(1))
table[[f"rank_{c}" for c in value_cols]]

In [ ]:
pd.DataFrame({c: s.price_of_conflict(table, c) for c in value_cols if c != "hub"}).T

## 4. Inspect in results-workbench

In [ ]:
wb = export_workbench_folder(
    cases.to_dict("records"), data.paths.results, data.paths.workbench,
    baseline=BASE, entities_csv=ws.reference_dir(data.paths) / "entities.csv",
)
print("Run the VS Code task 'Study: results viewer', choose 'Autogration export folder' and paste:", wb)